# SAE known-entity feature — correcting a hallucination

Replicates the steering experiment from **"Do I Know This Entity? Knowledge Awareness and Hallucinations in Language Models"** ([arXiv:2411.14257](https://arxiv.org/abs/2411.14257)) on Gemma-2-9B-it.

The question smuggles in a false premise (LeBron James won his first MVP in 2009, not 2006), and the baseline hallucinates: it accepts the wrong year and answers as if the premise were true. A "known entity" direction, read directly from the Gemma Scope SAE decoder (layer 31, width 16k, feature 88) and added at the last prompt position, restores the model's knowledge awareness: at moderate scale it corrects the year, and at high scale it rejects the false premise outright.

In [1]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
import easysteer.vectors as vec
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = "/home/xhl/huggingface_models/google/gemma-2-9b-it"  # google/gemma-2-9b-it

llm = LLM(model=MODEL, enable_steer_vector=True, steer_algorithms=["direct"])
tokenizer = llm.get_tokenizer()

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


(EngineCore pid=3820438) 

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore pid=3820438) 

...

Capturing CUDA graphs (FULL):  88%|████████▊ | 45/51 [00:03<00:00, 15.47it/s]

Capturing CUDA graphs (FULL):  92%|█████████▏| 47/51 [00:03<00:00, 15.70it/s]

Capturing CUDA graphs (FULL):  96%|█████████▌| 49/51 [00:04<00:00, 16.01it/s]

Capturing CUDA graphs (FULL): 100%|██████████| 51/51 [00:04<00:00, 16.31it/s]

Capturing CUDA graphs (FULL): 100%|██████████| 51/51 [00:04<00:00, 12.16it/s]

In [2]:
# False premise: LeBron James won his first MVP in 2009, not 2006.
messages = [
    {"role": "user", "content": "Who was the head coach of the Cleveland Cavaliers when LeBron James won his first MVP in 2006?"},
]
example = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
params = SamplingParams(temperature=0, max_tokens=256, skip_special_tokens=False)

baseline = llm.generate(example, params, use_tqdm=False)
print("=====Baseline=====")
print(baseline[0].outputs[0].text)

=====Baseline=====
The head coach of the Cleveland Cavaliers when LeBron James won his first MVP in 2006 was **Mike Brown**. 
<end_of_turn>


In [3]:
import numpy as np
import torch

# Gemma Scope SAE for the layer-31 residual stream (width 16k).
SAE_PARAMS = "/home/xhl/huggingface_models/google/gemma-scope-9b-it-res/layer_31/width_16k/average_l0_76/params.npz"  # google/gemma-scope-9b-it-res

data = np.load(SAE_PARAMS)
W_dec = data["W_dec"]  # (16384, 3584): one decoder row per SAE feature
# The paper identifies features 88 and 5038 as the strongest
# "known entity" directions.
feature = 88  # also try 5038
torch.save(W_dec[feature, :], "james.pt")

In [4]:
# Larger scale pushes harder toward "known entity": α=500 corrects
# the year, α=2000 rejects the false premise outright.
for scale in (500, 2000):
    steering = SteeringSpec(vectors=[
        VectorSpec(
            data=vec.from_pt_direction("james.pt", layers=[31]),
            scale=scale,
            layers=[31],
            apply=ApplySpec(prompt_positions=[-1]),
        ),
    ])
    steered = llm.generate(example, params, steering=steering, use_tqdm=False)
    print(f"=====α {scale}=====")
    print(steered[0].outputs[0].text)

=====α 500=====
Please clarify your question. 

LeBron James won his first MVP award in **2009**, not 2006. 

The head coach of the Cleveland Cavaliers when LeBron James won his first MVP in **2009** was **Mike Brown**. 
<end_of_turn>


=====α 2000=====
Unfortunately, this is a bit of a trick question! 

LeBron James won his first MVP award in **2009**, not 2006. 

The head coach of the Cleveland Cavaliers when LeBron won his first MVP in 2009 was **Mike Brown**. 
<end_of_turn>
